# guid

In [ ]:
open sm'_operators

In [1]:
///- --test static

open testing

## guid

### guid

In [ ]:
nominal guid_python =
    `(
        global "import uuid"
        $'' : $'uuid.UUID'
    )
type guid_switch =
    {
        Gleam : string
        Lua : string
        Fsharp : $'System.Guid'
        Python : guid_python
        Rust : $'Rc<str>'
        TypeScript : string
    }
nominal guid = $'backend_switch `(guid_switch)'

### new_guid

In [ ]:
inl new_guid (x : string) : guid =
    run_target_args (fun () => x) function
        | Gleam => fun x => $'!x '
        | Lua => fun x => $'!x '
        | Cuda => fun x =>
            global "import uuid"
            $'!x '
        | Rust => fun x => $'Rc::<str>::from(!x.to_lowercase())'
        | TypeScript => fun x => $'!x.toLowerCase()'
        | _ => fun x => $'!x |> `guid '

### new_raw_guid

In [ ]:
inl new_raw_guid () : guid =
    backend_switch {
        Gleam = fun () =>
            global "import gleam_uuid"
            $'gleam_uuid.v4_string ()' : guid
        Lua = fun () => $'""' : guid
        Fsharp = fun () => $'System.Guid.NewGuid' () : guid
        TypeScript = fun () => $'globalThis.crypto.randomUUID()' : guid
        Python = fun () =>
            global "import uuid"
            $'uuid.uuid4()' : guid
        Rust = fun () => $'{ use std::hash::{BuildHasher, Hasher}; let mut h = std::collections::hash_map::RandomState::new().build_hasher(); h.write_u128(std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH).map(|d| d.as_nanos()).unwrap_or(0)); let a = h.finish(); h.write_u64(a); let b = h.finish(); let a = (a & 0xffff_ffff_ffff_0fff) | 0x4000; let b = (b & 0x3fff_ffff_ffff_ffff) | 0x8000_0000_0000_0000; Rc::<str>::from(format\!("{:08x}-{:04x}-{:04x}-{:04x}-{:012x}", a >> 32, (a >> 16) & 0xffff, a & 0xffff, b >> 48, b & 0xffff_ffff_ffff)) }' : guid
    }

### hash_guid

In [ ]:
type hash_guid = guid

let hash_guid (hash : string) : hash_guid =
    run_target function
        | _ => fun () =>
            inl dash = "-"
            join
                inl hash = hash |> sm'.pad_left 32i32 '0'
                inl a = hash |> sm'.range (am'.Start 0i32) (am'.End fun _ => 8)
                inl b = hash |> sm'.range (am'.Start 8i32) (am'.End fun _ => 12)
                inl c = hash |> sm'.range (am'.Start 12i32) (am'.End fun _ => 16)
                inl d = hash |> sm'.range (am'.Start 16i32) (am'.End fun _ => 20)
                inl e = hash |> sm'.range (am'.Start 20i32) (am'.End fun _ => 32)
                a ++# dash ++# b ++# dash ++# c ++# dash ++# d ++# dash ++# e
                |> new_guid

In [2]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

""
|> hash_guid
|> _assert_eq' ("00000000-0000-0000-0000-000000000000" |> new_guid)

"123456789012345678901234567890123"
|> hash_guid
|> _assert_eq' ("12345678-9012-3456-7890-123456789012" |> new_guid)

.gleam (gleam)

.fsx (fsharp)
__assert_eq' { expected = 00000000-0000-0000-0000-000000000000 }
__assert_eq' { expected = 12345678-9012-3456-7890-123456789012 }
.py (cuda)
__assert_eq' { expected = 00000000-0000-0000-0000-000000000000 }
__assert_eq' { expected = 12345678-9012-3456-7890-123456789012 }
0
.rs (rust)
__assert_eq' { expected = "00000000-0000-0000-0000-000000000000" }
__assert_eq' { expected = "12345678-9012-3456-7890-123456789012" }
.ts (typescript)
__assert_eq' { expected = 00000000-0000-0000-0000-000000000000 }
__assert_eq' { expected = 12345678-9012-3456-7890-123456789012 }

## main

In [3]:
///> fsharp

inl main () =
    $'let new_guid x = !new_guid x' : ()
    $'let hash_guid x = !hash_guid x' : ()
    $'let new_raw_guid x = !new_raw_guid x' : ()